In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit.Chem import MolFromSmiles
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split
from pathlib import Path
from sklearn.model_selection import train_test_split
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from sklearn.model_selection import KFold
from chemprop import data, featurizers, models, nn
from rdkit.Chem import MolFromSmiles
import concurrent.futures
import logging
from chemprop.data import datapoints, dataloader, MoleculeDataset
from chemprop.featurizers import SimpleMoleculeMolGraphFeaturizer
from chemprop.data.datapoints import MoleculeDatapoint
import torch
import chemprop.nn.metrics as chem_metrics
from chemprop.nn.agg import (
    MultiHeadAttentiveAggregation,
    GatedAttentiveAggregation,
    GatedAttentiveAggregationv2
)
from assets.functionchem import *
from assets.chempropcombination import *

df = pd.read_csv("./data/clearancehuman.csv")
df = df.rename(columns={"HLM CLint": "CLint"})

df['scaffold'] = df['smiles'].apply(compute_scaffold)

# Step 1: Get the unique scaffolds
scaffold_list = df['scaffold'].unique()

# Step 2: Split the list of scaffolds into train, test, and validation
train_scaffolds, temp_scaffolds = train_test_split(scaffold_list, test_size=0.2, random_state=42)
test_scaffolds, val_scaffolds = train_test_split(temp_scaffolds, test_size=0.5, random_state=42)

# Step 3: Create the train, test, and validation sets by filtering the original DataFrame based on scaffold
train_df = df[df['scaffold'].isin(train_scaffolds)]
test_df = df[df['scaffold'].isin(test_scaffolds)]
val_df = df[df['scaffold'].isin(val_scaffolds)]

# Step 4: Verify the distribution of scaffolds in each set
print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")
print(f"Validation set size: {len(val_df)}")

train_df["logCL"] = np.log10(train_df["CLint"].clip(lower=1e-8))
val_df["logCL"] = np.log10(val_df["CLint"].clip(lower=1e-8))
test_df["logCL"] = np.log10(test_df["CLint"].clip(lower=1e-8))

results_df = run_chemprop_mp_agg_benchmark(
    train_df,
    val_df,
    test_df,
    target_column="logCL",
    max_epochs=200
)

Seed set to 42


Training set size: 2870
Test set size: 359
Validation set size: 359


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 4070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.



Training: BondMP3 + Mean


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    600 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

...

Output()

`Trainer.fit` stopped: `max_epochs=200` reached.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

Test: {'Target': 'logCL', 'MessagePassing': 'AtomMP6', 'Aggregation': 'MultiHead12', 'Test_MAE': 0.3307844400405884, 'Test_RMSE': 0.4847719371318817, 'Test_R2': 0.5220253467559814}

Final comparison:
   Target MessagePassing     Aggregation  Test_MAE  Test_RMSE   Test_R2
17  logCL        BondMP5     MultiHead12  0.318225   0.458490  0.572448
9   logCL        BondMP4      MultiHead4  0.320181   0.466975  0.556475
21  logCL        BondMP6      MultiHead4  0.322321   0.465479  0.559312
11  logCL        BondMP4     MultiHead12  0.323105   0.471570  0.547704
12  logCL        BondMP5            Mean  0.323921   0.481730  0.528005
13  logCL        BondMP5  GatedAttentive  0.324588   0.474806  0.541476
20  logCL        BondMP6      Attentive1  0.324896   0.476550  0.538102
40  logCL        AtomMP5      MultiHead8  0.325457   0.478823  0.533685
8   logCL        BondMP4      Attentive1  0.325458   0.470132  0.550458
22  logCL        BondMP6      MultiHead8  0.325836   0.469967  0.550775
23  logC